In [9]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

In [10]:
spark = SparkSession.builder.master("local[*]").appName('test').getOrCreate()

In [12]:
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [14]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [15]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [16]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [19]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [20]:
!wc -l head.csv

1001 head.csv


In [22]:
import pandas as pd

In [23]:
df_pandas = pd.read_csv('head.csv')

In [24]:
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [27]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [30]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [31]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [32]:
df.head(10)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [35]:
df = df.repartition(24)

In [36]:
df.write.parquet('fhvhv/2021/01/')

In [37]:
df = spark.read.parquet('fhvhv/2021/01/')

In [39]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [40]:
df.select("pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID")

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [41]:
df.select("pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID")\
    .filter(df.hvfhs_license_num == 'HV003')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [42]:
df.select("pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID") \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 23:41:01|2021-01-02 23:44:14|          20|          20|
|2021-01-01 17:38:33|2021-01-01 17:42:16|         254|         254|
|2021-01-01 02:31:50|2021-01-01 02:42:34|         173|         260|
|2021-01-01 22:59:30|2021-01-01 23:13:00|          49|         226|
|2021-01-01 09:34:40|2021-01-01 09:38:49|         212|         213|
|2021-01-01 07:02:50|2021-01-01 07:20:06|         235|         116|
|2021-01-02 20:16:11|2021-01-02 20:35:53|         225|          97|
|2021-01-01 13:44:14|2021-01-01 13:56:07|          60|          51|
|2021-01-02 16:09:08|2021-01-02 16:31:30|         167|         116|
|2021-01-01 02:31:13|2021-01-01 02:40:44|          32|          20|
|2021-01-02 15:15:55|2021-01-02 15:38:27|         236|         246|
|2021-01-02 13:02:09|2021-01-02 13:12:41|       

In [43]:

df \
    .withColumn("pickup_date", F.to_date(df.pickup_datetime)) \
    .withColumn("dropoff_date", F.to_date(df.dropoff_datetime)) \
    .select("pickup_date","dropoff_date","PULocationID","DOLocationID") \
    .show()   

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-01|  2021-01-01|         163|          48|
| 2021-01-01|  2021-01-01|         117|         201|
| 2021-01-02|  2021-01-02|          20|          20|
| 2021-01-01|  2021-01-01|         254|         254|
| 2021-01-01|  2021-01-01|         173|         260|
| 2021-01-01|  2021-01-01|          49|         226|
| 2021-01-01|  2021-01-01|         212|         213|
| 2021-01-01|  2021-01-01|         235|         116|
| 2021-01-02|  2021-01-02|         225|          97|
| 2021-01-01|  2021-01-01|          60|          51|
| 2021-01-02|  2021-01-02|         167|         116|
| 2021-01-01|  2021-01-01|          32|          20|
| 2021-01-02|  2021-01-02|         181|         249|
| 2021-01-01|  2021-01-01|         151|          24|
| 2021-01-02|  2021-01-02|         236|         246|
| 2021-01-02|  2021-01-02|         188|       

In [46]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [56]:

crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [59]:

df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID')

DataFrame[base_id: string, pickup_date: date, dropoff_date: date, PULocationID: int, DOLocationID: int]